In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sparsemax import Sparsemax


In [37]:
documents ={
    0: "The capital of France is Paris",
    1: "Python is a popular programming language",
    2: "The Earth revolves around the Sun",
    3: "Neural networks are used in deep learning",
    4: "Tokyo is the capital of Japan"
}

training_data = [
    ("What is the capital of France", 0),
    ("Which city is the capital of France", 0),
    ("Tell me about Paris", 0),

    ("What is Python", 1),
    ("Which programming language is popular", 1),
    ("Tell me about Python programming", 1),

    ("What does the Earth revolve around", 2),
    ("Which planet orbits the Sun", 2),

    ("What are neural networks used for", 3),
    ("Explain deep learning networks", 3),

    ("What is the capital of Japan", 4),
    ("Which city is Japan's capital", 4),
]
validation_data = [
    ("Is Paris the capital of France", 0),
    ("Tell me about the capital of France", 0),
    ("What is a popular programming language", 1),
    ("Tell me about the Python language", 1),
    ("What revolves around the Sun", 2),
    ("Tell me about the Earth", 2),
    ("What is deep learning", 3),
    ("Tell me about neural networks", 3),
    ("Is Tokyo the capital of Japan", 4),
    ("Which city is the capital of Japan", 4),
]


In [8]:
def tokenize(text):  #basic whitespace tokenizer
    return text.lower().replace("?","").split()

vocabulary =set()

for query, _ in training_data:
    vocabulary.update(tokenize(query))

for document in documents.values():
    vocabulary.update(tokenize(document))

word2idx = { word: i+1 for i,word in enumerate(sorted(vocabulary)) }

vocab_sz = len(vocabulary) + 1


In [9]:
def encode(text):
    return torch.tensor( [word2idx[w] for w in tokenize(text)], dtype=torch.long)

- Embedding layer
- Mean pooling to create a query representation
- MLP with one hidden layer

In [17]:
class DSI(nn.Module):
    def __init__(self, vocab_sz, n_embed, n_hidden, n_docs):
        super().__init__()

        self.embedding = nn.Embedding( vocab_sz, n_embed)
        self.fc1 = nn.Linear(n_embed, n_hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(n_hidden, n_docs)

    def forward(self, query):
        x = self.embedding(query)
        #x = x.mean(dim=1)
        mask = (query !=0).unsqueeze(-1)    #filter out padding tokens
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        
        return self.fc2(self.relu(self.fc1(x)))

In [44]:
model = DSI(vocab_sz, 32, 64, len(documents))
criterion= nn.CrossEntropyLoss()

In [14]:
[encode(q) for q, _ in training_data]

[tensor([36, 14, 33,  5, 22, 12]),
 tensor([37,  6, 14, 33,  5, 22, 12]),
 tensor([32, 19,  2, 24]),
 tensor([36, 14, 28]),
 tensor([37, 27, 17, 14, 26]),
 tensor([32, 19,  2, 28, 27]),
 tensor([36,  8, 33,  9, 29,  4]),
 tensor([37, 25, 23, 33, 31]),
 tensor([36,  3, 21, 20, 35, 11]),
 tensor([10,  7, 18, 20]),
 tensor([36, 14, 33,  5, 22, 15]),
 tensor([37,  6, 14, 16,  5])]

In [41]:
from torch.nn.utils.rnn import pad_sequence

train_X = pad_sequence(
    [encode(query) for query, _ in training_data],
    batch_first = True,
    padding_value=0
)
train_y = torch.tensor([doc_id for _, doc_id in training_data])

valid_X =pad_sequence(
    [encode(query) for query, _ in validation_data],
    batch_first = True,
    padding_value=0
)
valid_y = torch.tensor([doc_id for _, doc_id in validation_data])

In [ ]:
import importlib
import harness
importlib.reload(harness)
#harness.Trainer?

In [ ]:
learn = harness.Trainer(model, (train_X, train_y), (valid_X,valid_y), criterion=criterion,checkpoint_path=None, batch_size=8)

In [ ]:
learn.train_one_cycle(n_epochs=40, max_lr=1e-2, early_stopping=True)

In [47]:
@torch.no_grad()
def search(query):
    model.eval()
    tokens = encode(query).unsqueeze(0).to(learn.device)
    scores = model(tokens)
    doc_id = scores.argmax(dim=1).item()
    return documents[doc_id]

In [48]:
search("Which city is the capital of France?")

'The capital of France is Paris'